## Exercise 11 - A  Simple Convolutional Neural Network

Estimated time: **30-35 minutes**

The goal of this exercise is to implement a simple CNN for categorizing the notMNIST dataset (the letters A-J with various exotic typefaces). You will start with the non-convolutional network from the previous exercise, then modify it by adding convolutional and pooling layers as you attempt to improve the accuracy.

- Use **T4 GPU** as the hardware accelerator for this exercise

First run the code below to read in the notMNIST dataset.

In [ ]:
!wget https://pdl-doulos.s3.us-west-2.amazonaws.com/notMNIST.pickle

In [ ]:
%matplotlib inline
import pickle
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.utils import to_categorical

with open('notMNIST.pickle', 'rb') as file:
    data = pickle.load(file)
    train_dataset = data['train_dataset']
    train_labels = data['train_labels']
    valid_dataset = data['valid_dataset']
    valid_labels = data['valid_labels']
    test_dataset  = data['test_dataset']
    test_labels  = data['test_labels']

n_labels = 10

train_dataset = train_dataset.reshape(-1, 784).astype(np.float32)
valid_dataset = valid_dataset.reshape(-1, 784).astype(np.float32)
test_dataset  = test_dataset.reshape(-1, 784).astype(np.float32)
train_labels = to_categorical(train_labels)
valid_labels = to_categorical(valid_labels)
test_labels  = to_categorical(test_labels)

n_train = train_dataset.shape[0]
n_valid = valid_dataset.shape[0]
n_test = test_dataset.shape[0]

Build and run a neural network model with two hidden layers to establish a baseline for improvement:

In [ ]:
import tensorflow
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

image_size     =  28   # Width and height of image in pixels
n_full_units   = 128   # Number of units in each hidden layer
minibatch_size = 100   # Size of the minibatch used for stochastic gradient descent

def build_and_run_graph(n_epochs):
    tensorflow.keras.backend.clear_session()

    model = Sequential()
    model.add(Input(shape=(image_size*image_size,)))
    model.add(Dense(units=n_full_units, activation='relu'))
    model.add(Dense(units=n_full_units, activation='relu'))
    model.add(Dense(units=n_labels, activation='softmax'))
    model.summary()

    from tensorflow.keras.optimizers import Adam
    model.compile(loss='categorical_crossentropy', optimizer='Adam', metrics=['accuracy'])

    model.fit(train_dataset, train_labels, epochs=n_epochs, batch_size=minibatch_size, shuffle=True,
              validation_data=(valid_dataset, valid_labels))

    loss_and_acc = model.evaluate(test_dataset, test_labels, batch_size=minibatch_size, verbose=0)
    print(f'Test loss = {loss_and_acc[0]:6.3f}, accuracy = {loss_and_acc[1]*100:4.1f}')

In [ ]:
build_and_run_graph(n_epochs = 10)

Now add convolution and max pooling layers to the network. You will need to add a convolution layer (you choose the patch size, the stride, and the number of convolution filters), a max pooling layer (again choosing the patch size and stride), and a flattening layer. You will need to modify the first fully-connected layer (and its weights) to receive the output of the flattened layer.

You will need to run the following code to reshape the datasets from vectors to 28x28 pixel images, that is, from shape = (size, 784, 1) to shape = (size, 28, 28, 1)

In [ ]:
image_size       = 28   # Width and height of image in pixels
n_input_channels =  1   # The number of input channels, e.g. 3 for RGB

if (train_dataset.ndim == 2):
    train_dataset = train_dataset.reshape(-1, image_size, image_size, n_input_channels)
    valid_dataset = valid_dataset.reshape(-1, image_size, image_size, n_input_channels)
    test_dataset = test_dataset.reshape(-1, image_size, image_size, n_input_channels)

### Solution 

Here is our answer. Do not run the cell below unless you want to see the answer we provide!

<details>
    <summary> See our answer</summary>
    
   
    import tensorflow
    from tensorflow.keras.models import Sequential

    from tensorflow.keras.layers import Dense, Conv2D, MaxPooling2D, Flatten, Input

    n_full_units   = 128   # Number of units in each hidden layer

    minibatch_size = 100   # Size of the minibatch used for stochastic gradient descent

    def build_and_run_graph(n_epochs):
        tensorflow.keras.backend.clear_session()

        n_input_channels =  1   # The number of input channels, e.g. 3 for RGB
        patch_size       =  5   # Height and width of convolution patch in pixels
        pool_size        =  2
        n_features       = 30   # Number of feature maps in each convolution layer

        model = Sequential()

        model.add(Input(shape=(image_size, image_size, n_input_channels)))
        model.add(Conv2D(n_features, patch_size,
                     padding='same', activation='relu'))
        model.add(MaxPooling2D(pool_size=pool_size))
        model.add(Flatten())
        model.add(Dense(units=n_full_units, activation='relu'))
        model.add(Dense(units=n_full_units, activation='relu'))
        model.add(Dense(units=n_labels, activation='softmax'))
        model.summary()

        model.compile(loss='categorical_crossentropy', optimizer='Adam', metrics=['accuracy'])

        model.fit(train_dataset, train_labels, epochs=n_epochs, batch_size=minibatch_size, shuffle=True,
              validation_data=(valid_dataset, valid_labels))

        loss_and_acc = model.evaluate(test_dataset, test_labels, batch_size=minibatch_size, verbose=0)
        print(f'Test loss = {loss_and_acc[0]:6.3f}, accuracy = {loss_and_acc[1]*100:4.1f}')

    build_and_run_graph(n_epochs = 10)

    
</details>